In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive_output, FloatSlider, RadioButtons, HBox, VBox, Layout
from IPython.display import display, Markdown, HTML

display(HTML("""
<style>
.horizontal-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    gap: 25px !important;
}
.horizontal-radio .widget-radio-box label {
    margin: 0 !important;
    white-space: nowrap !important;
    font-weight: bold;
}
.horizontal-radio > label {
    display: none !important;
}
</style>
"""))

def plot_complete_lti_response(selection, r_val, theta_val):
    omega = np.linspace(0, np.pi, 1000)
    total_mag_dB = np.zeros_like(omega)
    total_phase = np.zeros_like(omega)
    total_gd = np.zeros_like(omega)
    th = np.deg2rad(theta_val)

    if selection == 'Single Zero':
        total_mag_dB += 10 * np.log10(1 + r_val**2 - 2 * r_val * np.cos(omega - th))
        total_phase += np.arctan2(r_val * np.sin(omega - th), 1 - r_val * np.cos(omega - th))
        total_gd += (r_val**2 - r_val * np.cos(omega - th)) / (1 + r_val**2 - 2 * r_val * np.cos(omega - th))

    elif selection == 'Single Pole':
        total_mag_dB -= 10 * np.log10(1 + r_val**2 - 2 * r_val * np.cos(omega - th))
        total_phase -= np.arctan2(r_val * np.sin(omega - th), 1 - r_val * np.cos(omega - th))
        total_gd -= (r_val**2 - r_val * np.cos(omega - th)) / (1 + r_val**2 - 2 * r_val * np.cos(omega - th))

    elif selection == 'Complex Conjugate Zeros':
        term1_mag = 10 * np.log10(1 + r_val**2 - 2 * r_val * np.cos(omega - th))
        term2_mag = 10 * np.log10(1 + r_val**2 - 2 * r_val * np.cos(omega + th))
        total_mag_dB += term1_mag + term2_mag

        term1_ph = np.arctan2(r_val * np.sin(omega - th), 1 - r_val * np.cos(omega - th))
        term2_ph = np.arctan2(r_val * np.sin(omega + th), 1 - r_val * np.cos(omega + th))
        total_phase += term1_ph + term2_ph

        term1_gd = (r_val**2 - r_val * np.cos(omega - th)) / (1 + r_val**2 - 2 * r_val * np.cos(omega - th))
        term2_gd = (r_val**2 - r_val * np.cos(omega + th)) / (1 + r_val**2 - 2 * r_val * np.cos(omega + th))
        total_gd += term1_gd + term2_gd

    elif selection == 'Complex Conjugate Poles':
        term1_mag = 10 * np.log10(1 + r_val**2 - 2 * r_val * np.cos(omega - th))
        term2_mag = 10 * np.log10(1 + r_val**2 - 2 * r_val * np.cos(omega + th))
        total_mag_dB -= term1_mag + term2_mag

        term1_ph = np.arctan2(r_val * np.sin(omega - th), 1 - r_val * np.cos(omega - th))
        term2_ph = np.arctan2(r_val * np.sin(omega + th), 1 - r_val * np.cos(omega + th))
        total_phase -= term1_ph + term2_ph

        term1_gd = (r_val**2 - r_val * np.cos(omega - th)) / (1 + r_val**2 - 2 * r_val * np.cos(omega - th))
        term2_gd = (r_val**2 - r_val * np.cos(omega + th)) / (1 + r_val**2 - 2 * r_val * np.cos(omega + th))
        total_gd -= term1_gd + term2_gd

    if selection == 'Single Zero':
        edu_text = r"""
* **Single Zero Effect ($z = re^{j\vartheta}$):**
  - **Amplitude:** Creates a local minimum (dip/notch) at $\omega = \vartheta$ reaching $20\log_{10}|1-r|$, and a maximum at $\omega = \pi + \vartheta$.
  - **Phase & Group Delay:** Contributes positive phase slope and smooth variations in $\tau(\omega)$.
"""
    elif selection == 'Single Pole':
        edu_text = r"""
* **Single Pole Effect ($p = re^{j\vartheta}$):**
  - **Amplitude:** Acts inversely to a zero, creating a resonance peak (maximum) near $\omega = \vartheta$ as $r \to 1$.
  - **Phase & Group Delay:** Contributes negative phase accumulation and positive group delay bumps.
"""
    elif selection == 'Complex Conjugate Zeros':
        edu_text = r"""
* **Complex Conjugate Zeros Effect ($z_{1,2} = re^{\pm j\vartheta}$):**
  - **Amplitude:** Combines two notch filters, producing deep attenuation (zeros near unit circle create sharp dips at $\omega = \vartheta$).
  - **Phase & Group Delay:** Doubled phase excursion and symmetric group delay contributions around the resonant frequencies.
"""
    else:
        edu_text = r"""
* **Complex Conjugate Poles Effect ($p_{1,2} = re^{\pm j\vartheta}$):**
  - **Amplitude:** Generates sharp resonance peaks (gain boost) when poles approach the unit circle ($r \to 1$).
  - **Phase & Group Delay:** Steep phase transitions and significant group delay variations ($\tau(\omega)$ peaks).
"""

    display(Markdown(edu_text))

    fig, axes = plt.subplots(3, 1, figsize=(9, 6.5), sharex=True)

    axes[0].plot(omega / np.pi, total_mag_dB, color='blue', lw=2)
    axes[0].set_ylabel('Amplitude (dB)', fontsize=10)
    axes[0].grid(True, linestyle='--', alpha=0.7)
    axes[0].set_title(f'LTI Frequency Response for: {selection}', fontsize=11, fontweight='bold')

    axes[1].plot(omega / np.pi, total_phase, color='green', lw=2)
    axes[1].set_ylabel('Phase (rad)', fontsize=10)
    axes[1].grid(True, linestyle='--', alpha=0.7)

    axes[2].plot(omega / np.pi, total_gd, color='red', lw=2)
    axes[2].set_ylabel(r'Group Delay $\tau(\omega)$', fontsize=10)
    axes[2].set_xlabel(r'Normalized Frequency ($\omega / \pi$)', fontsize=10)
    axes[2].grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

selection_widget = RadioButtons(
    options=['Single Zero', 'Single Pole', 'Complex Conjugate Zeros', 'Complex Conjugate Poles'],
    value='Complex Conjugate Zeros',
    description='',
    disabled=False,
    layout=Layout(width='100%')
)
selection_widget.add_class('horizontal-radio')

r_widget = FloatSlider(
    min=0.0,
    max=0.99,
    step=0.05,
    value=0.8,
    description='Radius (r):',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=Layout(width='20%')
)

theta_widget = FloatSlider(
    min=0.0,
    max=180.0,
    step=5.0,
    value=45.0,
    description='Angle (deg):',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=Layout(width='20%')
)

controls = VBox([
    HBox([selection_widget], layout=Layout(width='100%')),
    HBox([r_widget, theta_widget], layout=Layout(width='100%'))
])

out = interactive_output(
    plot_complete_lti_response,
    {'selection': selection_widget, 'r_val': r_widget, 'theta_val': theta_widget}
)

display(controls, out)